In [5]:
import re
import string
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier

In [9]:
# Path to your uploaded CSV
DATA_PATH = "cleaned_welfake_news_dataset.csv"  

TEXT_COL = "text"   # change if your dataset column name differs
LABEL_COL = "class" # change if your dataset column name differs

In [ ]:
# df = pd.read_csv(DATA_PATH)
df = pd.read_csv(DATA_PATH, encoding="latin-1")

# Drop rows with missing values in text or label
# Also drop rows with very short text (<=20 chars)
df = df.dropna(subset=[TEXT_COL, LABEL_COL])
df = df[df[TEXT_COL].str.len() > 20].reset_index(drop=True)
# Drop 20,000 random rows
# df = df.drop(df.sample(n=20000, random_state=42).index)


# Reset index after dropping
df = df.reset_index(drop=True)


# Light text cleaning
def clean_text(text):
    # s = str(s).lower()
    # s = re.sub(r"https?://\S+|www\.\S+", " ", s)  # remove URLs
    # s = re.sub(r"\s+", " ", s).strip()
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub("[%s]" % re.escape(string.punctuation), "", text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    text = re.sub(r"[^\w\s]", " ", text)  # Remove punctuation
    text = re.sub(r"\d+", " ", text)      # Remove numbers
    return text

df["text_clean"] = df[TEXT_COL].apply(clean_text)
df[LABEL_COL] = df[LABEL_COL].astype(int)
df.head()
print("Remaining rows:", len(df))


Remaining rows: 71238


In [12]:
# x=df['text']
x=df['text_clean']
y=df['class']

In [13]:
x

0        no comment is expected from barack obama membe...
1           did they post their votes for hillary already 
2         now  most of the demonstrators gathered last ...
3        a dozen politically active pastors came here f...
4        the rs  sarmat missile  dubbed satan   will re...
                               ...                        
71233    washington  reuters    hackers believed to be ...
71234    you know  because in fantasyland republicans n...
71235    migrants refuse to leave train at refugee camp...
71236    mexico city  reuters    donald trump s combati...
71237    goldman sachs endorses hillary clinton for pre...
Name: text_clean, Length: 71238, dtype: object

In [14]:
y

0        1
1        1
2        1
3        0
4        1
        ..
71233    0
71234    1
71235    0
71236    0
71237    1
Name: class, Length: 71238, dtype: int64

In [15]:
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.3)

In [16]:
x_train.shape

(49866,)

In [17]:
x_test.shape

(21372,)

In [18]:
vectorization=TfidfVectorizer(
    ngram_range=(1,3),   # unigrams + bigrams + trigrams
    max_features=20000,   # you can adjust this
    stop_words="english"
    )

In [19]:
xv_train=vectorization.fit_transform(x_train)

In [20]:
xv_test=vectorization.transform(x_test)

In [21]:
xv_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 9816622 stored elements and shape (49866, 20000)>

In [22]:
xv_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4235280 stored elements and shape (21372, 20000)>

In [23]:
from sklearn.linear_model import LogisticRegression

In [24]:
LR=LogisticRegression()

In [25]:
LR.fit(xv_train,y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [26]:
pred_lr=LR.predict(xv_test)

In [27]:
LR.score(xv_test,y_test)

0.9435242373198578

In [28]:
print(classification_report(y_test,pred_lr))

              precision    recall  f1-score   support

           0       0.95      0.93      0.94     10437
           1       0.94      0.95      0.95     10935

    accuracy                           0.94     21372
   macro avg       0.94      0.94      0.94     21372
weighted avg       0.94      0.94      0.94     21372



In [29]:
from xgboost import XGBClassifier

In [30]:
xgb=XGBClassifier(
    n_estimators=300,        # number of trees
    learning_rate=0.1,       # step size
    max_depth=6,             # tree depth
    subsample=0.8,           # random sampling for training
    colsample_bytree=0.8,    # random sampling for features
    eval_metric="logloss",   # avoid warnings
)

In [31]:
xgb.fit(xv_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [32]:
pred_xgb = xgb.predict(xv_test)

In [33]:
xgb.score(xv_test,y_test)

0.95709339322478

In [34]:
print(classification_report(y_test,pred_xgb))

              precision    recall  f1-score   support

           0       0.97      0.94      0.96     10437
           1       0.94      0.97      0.96     10935

    accuracy                           0.96     21372
   macro avg       0.96      0.96      0.96     21372
weighted avg       0.96      0.96      0.96     21372



In [ ]:
# below models are comment out because LR model perform better

In [35]:
DTC= DecisionTreeClassifier()  

In [36]:
DTC.fit(xv_train,y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [37]:
pred_dtc=DTC.predict(xv_test)

In [38]:
DTC.score(xv_test,y_test)

0.9136720943290286

In [39]:
print(classification_report(y_test,pred_dtc))

              precision    recall  f1-score   support

           0       0.92      0.90      0.91     10437
           1       0.91      0.93      0.92     10935

    accuracy                           0.91     21372
   macro avg       0.91      0.91      0.91     21372
weighted avg       0.91      0.91      0.91     21372



In [ ]:
# from sklearn.ensemble import RandomForestClassifier

In [ ]:
# rfc=RandomForestClassifier()

In [ ]:
# rfc.fit(xv_train,y_train)

In [ ]:
# pred_rfc=rfc.predict(xv_test)

In [ ]:
# rfc.score(xv_test,y_test)

In [ ]:
# print(classification_report(y_test,pred_rfc))

In [ ]:
# from sklearn.ensemble import GradientBoostingClassifier

In [ ]:
# gbc=GradientBoostingClassifier()

In [ ]:
# gbc.fit(xv_train,y_train)

In [ ]:
# pred_gbc=gbc.predict(xv_test)

In [ ]:
# gbc.score(xv_test,y_test)

In [ ]:
# print(classification_report(y_test,pred_gbc))

In [40]:
def output_label(n):
    if n==0:
        return "It's a Fake News"
    elif n==1:
        return "It's a Real News"

In [ ]:
# def manual_testing(news):
#     testing_news = {"text": [news]}
#     new_def_test = pd.DataFrame(testing_news)
#     new_def_test["text"] = new_def_test["text"].apply(clean_text)
#     new_x_test = new_def_test["text"]
#     new_xv_test = vectorization.transform(new_x_test)
    
#     pred_lr = LR.predict(new_xv_test)
#     # pred_dtc = DTC.predict(new_xv_test)
#     # pred_gbc = gbc.predict(new_xv_test)
#     # pred_rfc = rfc.predict(new_xv_test)

#     return "\n\nLR Prediction: {} \nXGB Prediction: {}".format(output_label(pred_lr[0]), output_label(pred_xgb[0])) # \n DTC Prediction: {} \n GBC Prediction: {} \n RFC Prediction: {}
#         # output_label(pred_dtc[0]),
#         # output_label(pred_gbc[0]),
#         # output_label(pred_rfc[0])


In [41]:
def manual_testing(news):
    # Validate input
    if not news or not isinstance(news, str):
        raise ValueError("Input news must be a non-empty string")
    
    # Prepare the input
    testing_news = {"text": [news]}
    new_def_test = pd.DataFrame(testing_news)
    new_def_test["text"] = new_def_test["text"].apply(clean_text)
    new_x_test = new_def_test["text"]
    new_xv_test = vectorization.transform(new_x_test)
    
    # Get predictions from both models
    pred_lr = LR.predict(new_xv_test)
    pred_xgb = xgb.predict(new_xv_test)  # Compute XGBoost prediction for the input
    pred_dtc = DTC.predict(new_xv_test)
    
    # Return formatted predictions
    return "\n\nLR Prediction: {} \nXGB Prediction: {} \n DTC Prediction: {}".format(
        output_label(pred_lr[0]), 
        output_label(pred_xgb[0]),
        output_label(pred_dtc[0])
    )

In [47]:
news_article=str(input("Enter a news article: "))
manual_testing(news_article)

"\n\nLR Prediction: It's a Real News \nXGB Prediction: It's a Real News \n DTC Prediction: It's a Real News"

In [43]:
# Save the best model (Logistic Regression performed best)
joblib.dump(xgb, "fake_news_model.pkl")

['fake_news_model.pkl']

In [44]:
# Save vectorizer
joblib.dump(vectorization, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [45]:
# Load back the model
model = joblib.load("fake_news_model.pkl")

# Load back the vectorizer
vectorizer = joblib.load("tfidf_vectorizer.pkl")

print("Model type:", type(model))
print("Vectorizer type:", type(vectorizer))

Model type: <class 'xgboost.sklearn.XGBClassifier'>
Vectorizer type: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
